# Qwen Image Edit Web Server (GGUF) — Google Colab

GGUF量子化版 Qwen-Image-Edit-Rapid を Flask Web サーバーとして起動し、cloudflared で外部公開します。

**必要環境:** GPU ランタイム (A100 以上推奨)

## 1. GPU確認

In [ ]:
!nvidia-smi

## 2. パッケージインストール

In [ ]:
!pip install -q flask pillow huggingface_hub transformers accelerate safetensors googletrans
!pip install -q "diffusers>=0.36.0,<0.37.0" gguf
!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

## 3. HuggingFace ログイン（任意）

ログインするとモデルダウンロードが高速化されます（レート制限緩和）。  
Colab サイドバーの 🔑「シークレット」に `HF_TOKEN` を登録しておくと自動認証されます。  
未登録の場合は `login()` でトークン入力画面が表示されます。スキップも可。

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("HuggingFace: Colab Secrets で認証しました")
except Exception:
    print("Colab Secrets に HF_TOKEN が未登録です。手動ログインを試みます...")
    print("スキップする場合はこのセルの出力を無視してください。")
    try:
        from huggingface_hub import login
        login()
    except Exception:
        print("HuggingFace: 未ログイン（匿名ダウンロード）")

## 4. Transformer config 準備

GGUF の `from_single_file` に必要な config.json を作成します。

In [ ]:
import json, os

config_dir = "qwen-image-edit-transformer-config"
os.makedirs(config_dir, exist_ok=True)

config = {
    "_class_name": "QwenImageTransformer2DModel",
    "_diffusers_version": "0.36.0",
    "attention_head_dim": 128,
    "axes_dims_rope": [16, 56, 56],
    "guidance_embeds": False,
    "in_channels": 64,
    "joint_attention_dim": 3584,
    "num_attention_heads": 24,
    "num_layers": 60,
    "out_channels": 16,
    "patch_size": 2
}

with open(os.path.join(config_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("config.json created")

## 5. app_gguf.py を配置

リポジトリから `server/app_gguf.py` をアップロードするか、以下でクローンします。

In [ ]:
import shutil

# --- 方法A: ファイルアップロード済みの場合 ---
# Google Colab のファイルブラウザ(左サイドバー)で app_gguf.py をアップロードしてください

# --- 方法B: リポジトリからコピー (URLは適宜変更) ---
!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_gguf.py .

# server/ ディレクトリ構造の場合 TRANSFORMER_CONFIG パスを調整
# app_gguf.py は親ディレクトリの qwen-image-edit-transformer-config を参照するため
# カレントディレクトリに app_gguf.py がある場合はシンボリックリンクで対応
os.makedirs("server", exist_ok=True)
if os.path.exists("app_gguf.py") and not os.path.exists("server/app_gguf.py"):
    shutil.copy("app_gguf.py", "server/app_gguf.py")

print("準備完了: server/app_gguf.py と qwen-image-edit-transformer-config/ を確認してください")
!ls -la server/app_gguf.py qwen-image-edit-transformer-config/config.json 2>/dev/null || echo "⚠ ファイルが見つかりません。app_gguf.py をアップロードしてください。"

## 6. 設定

パスワードやプリセットを変更できます。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 5000                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化
GGUF_FILE = "v23/Qwen-Rapid-NSFW-v23_Q8_0.gguf" # 空欄ならデフォルト (Q3_K) を自動ダウンロード
                            # 例: "v23/Qwen-Rapid-NSFW-v23_Q2_K.gguf" (低VRAM向け)

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Enhance quality and fix artifacts.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
]

# LoRA
LORAS = [
    "/tmp/LoRA/anything2real.safetensors",
]


## 7. サーバー起動 + cloudflared 公開

Flask サーバーをバックグラウンドで起動し、cloudflared で公開URLを生成します。  
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
import subprocess, time, re, threading

SERVER_LOG = "server.log"

# コマンドライン引数を構築
cmd = [
    "python", "-u", "server/app_gguf.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--password", PASSWORD,
]
if GALLERY:
    cmd.append("--gallery")
if GGUF_FILE:
    cmd.extend(["--gguf-file", GGUF_FILE])
for lora in LORAS:
    cmd.extend(["--lora", lora])
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"[colab] starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
print("[colab] waiting for model to load...")
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("[colab] ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\n[colab] server is running on port {PORT}")
print(f"[colab] starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
else:
    print("[colab] ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

## 8. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# サーバーログ確認（直近20行）
!tail -20 server.log

# サーバー + トンネル停止

```
server_proc.terminate()
tunnel_proc.terminate()
log_fh.close()
print("stopped")
```